In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from os import listdir
import matplotlib.pyplot as plt
from pendulibrary.continuation import find_bifurcation, adaptive_cont
from pendulibrary.targeter import dc_underconstrained
from pendulibrary.common_targetters import single_fixed
from pendulibrary.plotters import plot_nu_functions, compare_fast
from pendulibrary.integrate import integrate_state
from pendulibrary.common import hamiltonian
from tqdm.auto import tqdm

In [ ]:
# The P4as have something going on
# Go backward too

In [ ]:
fname = "DDsp-P4a"

data = np.load(f"../database/{fname}.npz")
x0s_in = data["x0s"]
periods_in = data["periods"]
tangents_in = data["tangents"]
eigs_in = data["eigs"]
Lr, Mr = data["params"]

targ = single_fixed(0, x0s_in[0][0], 3, Lr, Mr, 1e-14)
func = targ.g_dg_stm

In [ ]:
%matplotlib ipympl
fig = plot_nu_functions(eigs_in, 4)
plt.show()

In [ ]:
im1 = 61
im0 = 60

Xm0 = targ.get_X(x0s_in[im0], periods_in[im0])
Xm1 = targ.get_X(x0s_in[im1], periods_in[im1])

g, dg, stm = func(Xm1)
svd = np.linalg.svd(dg)
tangent = svd.Vh[-1]
sprev = np.linalg.norm(Xm0 - Xm1)
if np.dot(tangent, Xm0 - Xm1) / sprev < 0:
    tangent *= -1
sprev = np.linalg.norm(Xm0 - Xm1)
print(f"Last dist is {sprev:.3e}")

In [ ]:
X0 = find_bifurcation(
    Xm1,
    func,
    tangent,
    sprev / 5,
    targ_tol=1e-13,
    bisect_tol=1e-5,
    period_mult=1,
    debug=True,
    scale=5,
    seek_local_opt=False,
)

In [ ]:
X0

In [ ]:
X1 = X0.copy()
X1[-1] *= 1
g, dg, stm = func(X1)
svd = np.linalg.svd(dg)
tan = svd.Vh[-2]
svd.S

In [ ]:
cont_kwargs = dict(
    s0=5e-6, s_min=1e-6, S=7.0, tol=1e-7, max_iter=15, target_iter=10, rate=1.05
)
Xs, eig_vals, (DFs, tangents, stms) = adaptive_cont(
    X1, func, -tan, **cont_kwargs, exact_tangent=True
)

In [ ]:
fnames = [f.removesuffix(".npz") for f in listdir("../database") if f.endswith(".npz")]
x0s_new = np.array([targ.get_x0(X) for X in Xs])
periods_new = np.array([targ.get_period(X) for X in Xs])
compare_fast(periods_new, hamiltonian(x0s_new.T, Lr, Mr), fnames)

In [ ]:
plt.close("all")
ts, xs, fs = integrate_state(x0s_new[-1], periods_new[-1], Lr, Mr, 1e-14)
y = -np.cos(xs[0]) - Lr * np.cos(xs[1])
x = np.sin(xs[0]) + Lr * np.sin(xs[1])

plt.plot(x, y)
plt.axis("equal")
plt.show()

In [ ]:
np.savez_compressed(
    "../database/DDsp-P4a-B1",
    x0s=x0s_new.astype(np.float32),
    periods=periods_new.astype(np.float32),
    eigs=eig_vals.astype(np.complex64),
    tangents=tangents.astype(np.float32),
    hamiltonians=hamiltonian(x0s_new.T, Lr, Mr).astype(np.float32),
    params=np.array([Lr, Mr]),
)